# Recherche de parties d'échecs remarquables dans la base Lichess

Notebook principal

| Étape | Script | Résultat |
|---|---|---|
| 1. Filtrage du dump | `filtrer_parties.py` | parties retenues, avec leurs évaluations |
| 1. Features | `analyse_features.py` | une soixantaine de features par partie, statistiques et graphiques |
| 2. Sélection | `selection_pot.py` | quelques milliers de parties atypiques (Mahalanobis + POT, par code ECO) |
| 3a. Parties du mois | `scraping.py` | parties du mois Lichess (positifs) |
| 3b. Témoin | `temoin.py` | parties tirées au hasard (non étiquetées) |
| 3c. Augmentation | `augmentation.py` | mêmes évaluations Stockfish et mêmes features pour les trois groupes |
| 3d. PU learning | `pu_learning.py` | candidates étiquetées positives |
| 3e. Échantillon final | `echantillon_final.py` | 20 parties |
| Comparaison | `comparaison.py` | 200 parties quelconques (10 blocs de 20) |

**Utilisation.** Réglez la cellule de configuration, puis exécutez les cellules dans l'ordre. Une étape déjà faite
peut être sautée (`LANCER[...] = False`) : ses fichiers sont alors seulement relus et affichés. Pour vérifier
l'installation en quelques minutes, mettez `MODE = "exemple"` (petit jeu de données synthétique fourni).

## Configuration

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

%matplotlib inline
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)

RACINE = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent     # racine du dépôt
SRC = RACINE / "src"

MODE = "complet"            # "complet" : vos données ; "exemple" : jeu de données synthétique fourni (test rapide)

# --- Mode complet : à adapter
DONNEES = Path(os.environ.get("LICHESS_DATA", RACINE / "donnees"))     # dossier des données (hors dépôt git)
DUMP = "lichess_db_standard_rated_2026-01.pgn.zst"                    # dump mensuel, placé dans DONNEES
STOCKFISH = os.environ.get("STOCKFISH_PATH", "")                       # chemin de l'exécutable Stockfish
P = dict(tranche=500_000, bloc_lecture=100_000, n_min=5_000, m_min=50, p_seuil=1e-3,
         n_temoin=5_000, profondeur=12, seuil_pu=0.5, n_final=20, graine=0)

# --- Étapes à exécuter (False : relire et afficher les fichiers déjà produits)
LANCER = dict(filtrage=True, features=True, selection=True, scraping=True, temoin=True,
              augmentation=True, pu=True, echantillon=True, comparaison=True)

if MODE == "exemple":
    DONNEES = RACINE / "exemples" / "travail"
    DUMP = str(RACINE / "exemples" / "mini_dump.pgn.zst")
    P.update(tranche=100, bloc_lecture=200, n_min=50, m_min=20, p_seuil=0.1, n_temoin=200, profondeur=8)

DONNEES.mkdir(parents=True, exist_ok=True)
RES, SEL, PU = DONNEES / "resultats", DONNEES / "resultats" / "selection", DONNEES / "pu"
print(f"Mode : {MODE}\nDonnées : {DONNEES}\nDump : {DUMP}\nStockfish : {STOCKFISH or '(non renseigné)'}")

In [ ]:
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1", "MPLBACKEND": "Agg"}
if STOCKFISH:
    ENV["STOCKFISH_PATH"] = STOCKFISH


def lancer(script, *args):
    """Exécute src/<script> dans un processus séparé (comme dans un terminal) et affiche sa sortie en direct."""
    cmd = [sys.executable, "-u", str(SRC / script), *[str(a) for a in args]]
    print("$ python src/" + script + " " + " ".join(str(a) for a in args) + "\n")
    t = time.perf_counter()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", env=ENV, cwd=RACINE) as p:
        for ligne in p.stdout:
            print(ligne, end="")
    duree = time.perf_counter() - t
    if p.returncode:
        raise RuntimeError(f"{script} s'est arrêté avec une erreur (code {p.returncode}) : voir la sortie ci-dessus.")
    print(f"\n[{script} terminé en {duree / 60:.1f} min]")


def liens(df, **kw):
    """Tableau HTML où les adresses de parties sont cliquables."""
    display(HTML(df.to_html(render_links=True, escape=True, index=False, **kw)))


def images(*motifs, largeur=None):
    for motif in motifs:
        for f in sorted(Path(RES).glob(motif)) if not Path(motif).is_absolute() else [Path(motif)]:
            if f.exists():
                display(Markdown(f"`{f.name}`"))
                display(Image(filename=str(f), width=largeur))

## Étape 1 : filtrage du dump

Parties gardées : terminaison normale, résultat 1-0 ou 0-1, cadence définie, Elo des deux joueurs ≥ 1500 (écart ≤ 400),
20 à 120 coups, évaluation présente. Le dump est lu en flux (sans être décompressé sur le disque).

In [ ]:
if LANCER["filtrage"]:
    lancer("filtrer_parties.py", "--dossier", DONNEES, "--dump", DUMP, "--tranche", P["tranche"])

import pyarrow.parquet as pq
n_filtrees = pq.ParquetFile(DONNEES / "parties_filtrees.parquet").metadata.num_rows
print(f"Parties retenues : {n_filtrees:,}")
comptes = pd.read_csv(DONNEES / "comptes_par_tranche.csv")
if len(comptes) > 1:
    display(comptes["n_gardees"].describe().to_frame("parties gardées par tranche").T.round(1))
    ax = comptes.plot(x="tranche", y="n_gardees", figsize=(10, 3), legend=False, title="Parties gardées par tranche")
    ax.set_ylabel("parties gardées"); plt.show()

ensae-prog27

# Projet Python pour la data science : Les plus belles parties d'échec - trouver l'esthétisme par l'algorithme 


Auteurs : Camille BOURGET, Gautier COSTE, Valentin FAZEMBAI TREMOUILLES
---

# Sommaire

## I : Introduction

### A : Contexte
### B : Objectif du projet
### B : Architecture
### C : Résultats

##  II : Méthode et vue d'ensemble qualitative du projet

### A : Problème des aiguilles dans une botte de foin
### B : Présentation du dataset
### C : 'L'esthétisme' : comment quantifier un concept humain ?
### D : Reproductibulité des résultats
### E : Usage de l’IA générative

## III : Présentation détaillée des trois couches

### A : Nettoyage et premières hypothèses
### B : Vectorisation et élagage statistique : théorie des valeurs extrêmes
### C :

## IV : Conclusion

---

## Introduction

### A : Contexte

Chaque mois près de 100 millions de parties d'échecs sont jouées sur lichess.org, le deuxième site le plus populaire en la matière. Chaque mois les joueurs proposent et élisent au cours d'un vote la 'game of the month', à savoir la plus belle partie selon des critères humains subjectifs d'esthétisme : game of the month d'août 2026, https://lichess.org/@/Lichess/blog/lichess-game-of-the-month-august-26/ekVdAfDN.

### B : Objectif du projet

L'objectif du projet est de concevoir un algorithme prenant en entrée les 100 millions de parties mensuelles et renvoyant 20 parties esthétiques pour l'oeil humain qui pourraient être désignées 'game of the month'.

Au cours de ce travail nous concevrons cet algorithme et évaluerons ses performances.

### C : Architecture

Nous avons à faire à un problème de recherche d'aiguille dans une botte de foin. Ainsi, notre algorithme fonctionne comme un entonnoir : trois filtres successifs permettent de passer du dataset complet à l'échantillon final de 20 parties.

Plus le filtre traite un nombre important de parties plus il doit être léger en s'appuyant sur des opérations simples, et inversement plus il est proche de la sortie de l'entonnoir plus il peut utiliser des méthodes couteuses pour classifier les dernières parties.

La première couche  (partie III A) correspond au nettoyage des données et à la sélection des parties sur lesquelles nous souhaitons nous attarder permettant de passer de 100 à 3 millions de parties, la seconde (partie III B) utilise des propriétés statistiques de manière à passer de 3 millions à 3 000 parties, et la dernière utilise un modèle de Machine Learning pour obtenir 20 parties.

### D : Résultats




## II : Méthode et vue d'ensemble qualitative du projet

### A : Problème des aiguilles dans une botte de foin

Notre sujet est assez atypique, en effet en plus de réaliser des statistiques descriptives, le coeur de notre problématique est de concevoir un algorithme. Pour que le projet soit sérieux il convient de pouvoir évaluer ses performances. Définissons une métrique qui permet cela.

Cet algorithme tente de trouver des éléments rares (aiguilles) noyés dans un volume considérable d'information (la botte de foin). Ainsi nous utiliserons le taux de vrais positifs : le nombre de parties effectivement remarquables parmi les 20 renvoyées par l'algorithme.

Nous comparerons ce résultat au témoin servant de référence. Le témoin est obtenu en tirant au hasard 10 échantillons de 20 parties parmi les 100 millions de parties. Après avoir étudié la moyenne empirique et la distribution du témoin nous pourrons conclure que notre algorithme est meilleur (ou moins bon...) que le baseline random avec un certain intervalle de confiance.

Nota Bene : Faute de mieux, la classification binaire entre partie 'esthétique' et lambda a été faite à la main par un panel de 3 joueurs (classés 1200, 1500 et 2000 elo sur chess.com) à qui les 220 parties ont été montrées mélangées, une partie était 'esthétique' si au moins 2 des 3 joueurs humains la jugeaient telle quelle et lambda sinon.

### B : Présentation du dataset

Les données utilisées pour ce projet proviennent de la base de donnée Lichess : https://database.lichess.org/
Les données sont des parties au format PGN, le format standard pour enregistrer des parties d'échec. Pour chaque partie, le PGN contient : le tournoi ou contexte de la partie, le lieu, la date, la ronde, les noms de joueurs, le resultat le classement des joueurs et la cadence, ainsi que l'ensemble de la partie avec les coups en notation algébrique standart (SAN). Ce format permet de stocker un maximum de données sur une partie tout en étant très léger, ce qui nous a permis de travailler sur un grand nombre de parties (environ 100 millions de parties par mois).

### C : 'L'esthétisme' : comment quantifier un concept humain ?

Notre objectif étant de trouver les plus belles parties parmis celles jouées dans le mois, la première question était de définir ce qui fait une "belle" partie d'échecs. En effet, on part d'une notion subjective et humaine, qui ne peut être saisie par une machine. Il faut donc l'adapter avec des critères quantifiables et mesurables pour s'en approcher un maximum, et permettre de filtrer les nombreuses parties pour n'en conserver que quelques unes. Ainsi, nous avons selectionnés les éléments qui nous parraissaient les plus pertinents pour juger de l'esthétisme d'une partie d'échecs : les retournements d'avantages, les sacrifices de pièces, le dynamisme de la partie ... Nous avons ainsi dû selectionner nos critères de tri des parties, ainsi que la façon de les mesurer, afin de filtrer les parties au fur et à mesure, en commençant par les critères les plus évidents et facilement quantifiables pour ensuite étudier plus précisemment les parties restantes (voir partie II). 

### D - Reproductibilité des résultats

**Reproductibilité des chemins d’accès**

Les fichiers “filtrer_parties.py” et “analyse_features.py” comprennent la variable RACINE qui permet à n’importe quelle machine de retrouver le projet dans son ensemble, indépendamment de son emplacement en local. De même, la variable DOSSIER renvoie aux données directement depuis le projet. Les variables FICHIER et SORTIE permettent quant à elles de clarifier le parcours exact des fichiers (données filtrées puis analysées). Les deux scripts acceptent également des options en ligne de commande (--dossier, --dump, --fichier) qui adaptent les chemins d’entrée et sortie des fichiers directement lors de l’exécution. 

**Paramètres et filtres explicites**

Les règles appliquées, notamment afin de filtrer et évaluer les parties, sont regroupées en haut des fichiers pour ne laisser aucune ambiguïté. Dans “filtrer_parties.py” : COUP_MIN, COUP_MAX par exemple, ou dans “features_core.py” : VALEUR (vecteur des valeurs des pièces). 

**Calcul en flux constant**

Pour que le projet puisse être exécuté sur n'importe quelle machine sans saturer la mémoire vive (RAM), l'ensemble du traitement est conçu en flux continu. Dans “filtrer_parties.py”, le fichier compressé Lichess est lu au fil de l'eau (stream_reader) et écrit sur disque par lots de 100 000 parties (vider_lot), sans jamais être entièrement décompressé en mémoire. De la même manière, analyse_features.py traite les données par blocs (iter_batches) et met à jour les statistiques au fur et à mesure via la classe Accumulateur. 


### E - Usage de l’IA générative
Il convient enfin de détailler l’usage qui a été fait de l’IA générative pour réaliser ce projet. Avant tout, l’IA nous a permis de trouver rapidement des datasets adaptés à notre projet sur internet. Concernant le projet en lui-même, nous avons codé une première version permettant d’avoir un squelette déjà opérationnel sur un nombre de parties restreint. Cette première version manuelle était d’ailleurs nécessaire pour garder la main sur l’ensemble des critères choisis, qui visent in fine à détecter des “belles parties”, à l'aune de notre propre appréciation esthétique. Ce n’est que par la suite que nous avons employé Claude notamment, pour débugger certains blocs, et scaler notre projet afin de le rendre applicable à des millions de parties (ex : multiprocessing dans analyse_features.py). La rédaction du README a enfin été essentiellement manuelle pour s’assurer de la clarté et de la pertinence des explications au regard de nos propres impressions. 


---


## III : Présentation détaillée des trois couches

### A : Nettoyage et premières hypothèses

La première étape de notre projet consiste à réduire grandement l'échantillon de données pour ne conserver qu'un nombre restreint de parties intéressantes, que nous analyserons ensuite de manière plus poussée. 
Pour cela, nous regroupons les parties par paquets de 500 000, et nous effectuons le filtrage suivant : conserver seulement les parties avec un vainqueur, dont les deux joueurs ont plus de 1500 elo (ce qui correspond au top 50% des joueurs) et avec un faible écart de niveau entre les joueurs. Nous avons supprimons ensuite les parties jouées par correspondance, et celles dont l'évaluation n'est pas disponible : l'évaluation nous permet d'avoir plus d'informations sur la partie, et le fait qu'un des joueurs ait cherché à l'évaluer renvoie déjà un signal sur sa qualité. 
A la fin de cette étape, il nous reste un total de *3 124 123* parties, soit *3,3 %* du nombre du départ. Cette première étape nous permet donc de réduire grandement la quantité de données à traiter de manière rapide et efficace (on a peu de chance d'éliminer une des plus "belles" parties de cette façon).
Enfin, en plus de fluidifier l'exécution du code, le fait de regrouper les parties par groupe de 500 000 nous a permis d'évaluer combien de parties ce filtrage conserve en moyenne afin dimensionner au mieux l'étape suivante. En effet, nous avons constaté lors des tests que cette étape conserve en moyenne *158 000* parties par groupe, avec une faible variance empirique (*3%* de la moyenne). Sur cette base, nous avons donc calibré l'algorithme de l'étape suivante pour traiter 3 millions de parties, le risque de dépasser significativement ce nombre étant très faible. 

### B - Second filtrage et évaluation des parties 

La seconde étape correspond à l’analyse et l’évaluation des 3 millions de parties conservées. Le but est de distinguer une poignée de parties exceptionnelles.

D’abord, il s’agit de définir des features (critères d’évaluation) que l’on peut regrouper au sein des catégories suivantes : complexité de la partie (elo moyen des deux joueurs, longueur de la partie), combats et fluctuations matérielles (nombre de captures, remontada…), sacrifices et déséquilibre d’échanges, complexité tactique (nombre d’échecs, de coups avec capture, de promotions…), dynamisme de la partie (nombre de coups du roi, longueur max d’une séquence sans capture). Une cinquantaine de features sont calculées partie par partie dans features_core.py (fonctions features_partie et features_eval). Chaque partie est alors identifiée comme un vecteur de dimension du nombre de features.

Ensuite, il s’agit d’identifier les vecteurs, et de fait les parties, les plus rares. C’est une des grandes difficultés auxquelles nous avons été confrontés puisqu’il fallait déterminer une métrique pour classer les vecteurs, en s’assurant que celle-ci permettrait de distinguer des “belles parties” conformément à notre opinion sur l'esthétique d'une partie.

Pour ce faire, nous avons cherché à obtenir un score d’atypicité d’une partie à partir de sa distance au centre de gravité de nos 3 millions de vecteurs. Les métriques classiques (norme infinie, distance euclidienne) ne convenaient pas puisqu’elles ne prennent pas en compte l’échelle différente des variables (binaires, sur N, sur Z-...) et la corrélation entre ces variables (une partie longue a mécaniquement plus de captures par exemple). C’est pourquoi nous avons utilisé la distance de Mahalanobis qui corrige ces deux défauts. De plus, nous avons scindé les calculs par type d’ouverture (certaines ouvertures aboutissent à des parties plus longues par exemple). En outre, c’est une version robuste de la distance qui a été employée afin de limiter l’effet des valeurs extrêmes sur le centre de gravité. Précisément, dans elagage_evt.py, les parties sont d'abord regroupées par code ECO (groupes_reference), puis les variables sont transformées (transformer) et la distance de Mahalanobis robuste est calculée en écartant itérativement les 10 % de parties les plus extrêmes (ajuster avec TRIM = 0.9, puis distances).

Enfin, une fois que l’on a ces scores d’atypicité, il faut pouvoir en déduire quelles parties sont vraiment “exceptionnelles”. Par exemple, un top 100 (ou autre) n’a pas de justification statistique. C’est pourquoi nous avons utilisé l’EVT (Extreme Value Theory) via la méthode POT (Peaks Over Threshold). Cela consiste à calculer pour chaque partie la probabilité qu’une partie de cette même ouverture soit au moins aussi atypique que celle-ci (p-valeur de rareté). On peut alors dire des parties dont cette probabilité est inférieure à un seuil préalablement déterminé (0.1% par exemple) qu’elles sont véritablement exceptionnelles. 

## Justification statistique de la méthodologie

En amont du traitement des parties, nous avons observé les données par le biais de statistiques descriptives. Cela nous a permis d'employer les méthodes adaptées au dataset, mentionnées ci-dessus (Mahalanobis, découpage par ouverture, modélisation EVT/POT...). 
NB : ces statistiques ont été réalisées sur 200 000 parties (et environ 6000 après filtrage) du fait de difficultés à télécharger l'ensemble du dataset. 

1. **Corrélations entre features**
![Matrice de corrélation](images/matrice_correlation.png)
*Les fortes corrélations observées entre variables (notamment entre la longueur de la partie et les compteurs de captures ou d'échecs) prouvent qu'une distance euclidienne biaiserait le score et justifient l'usage de la distance de Mahalanobis.*

2. **Distributions asymétriques à queues longues**
![Distributions en échelle logarithmique](images/graph2_distributions.png)
*Les distributions fortement asymétriques (où l'immense majorité des parties reste proche de zéro tandis que quelques cas atteignent des valeurs extrêmes) justifient à la fois la transformation `log1p` et le recours à la théorie des valeurs extrêmes.*

3. **Différences selon l'ouverture (code ECO)**
![Comparaison entre ouvertures](images/graph3_comparaison_eco.png)
*Les écarts de distribution observés entre les principales ouvertures confirment que la « norme » dépend du type de partie joué, ce qui impose de calculer la distance de Mahalanobis et les seuils de rareté ouverture par ouverture.*

4. **Modélisation de la queue par EVT / POT**
![Ajustement EVT/POT](images/graph4_evt_pot.png)
*Au-delà du seuil du 95e percentile ($u$), la loi de Pareto généralisée modélise la queue des distances de Mahalanobis et permet d'isoler les parties statistiquement exceptionnelles sous le seuil $p \le 10^{-3}$.*


## Étape 1 (suite) : calcul des features

Une soixantaine de features par partie : matériel et structure (groupes A à E), puis évaluation du moteur (F à L).

In [ ]:
if LANCER["features"]:
    lancer("analyse_features.py", "--dossier", DONNEES, "--bloc-lecture", P["bloc_lecture"])

resume = pd.read_csv(RES / "resume_statistiques.csv", index_col=0)
display(resume.round(3))

In [ ]:
images("distributions_*.png")

In [ ]:
images("matrice_correlation.png", "nuages_de_points_materiel.png", "nuages_de_points_evaluation.png")

**Commentaires :**

*(vos observations ici)*

## Étape 2 : sélection des parties atypiques (Mahalanobis + POT)

Pour chaque code ECO comptant au moins `n_min` parties (5 000) : distance de Mahalanobis robuste de chaque partie
au centre de son ouverture, loi de Pareto généralisée ajustée sur la queue (méthode POT), p-valeur de rareté.
Les parties retenues de tous les codes ECO sont réunies dans une seule base, `base_selection.parquet`.

In [ ]:
if LANCER["selection"]:
    lancer("selection_pot.py", "--dossier", DONNEES, "--n-min", P["n_min"], "--m-min", P["m_min"],
           "--p-seuil", P["p_seuil"])

resume_eco = pd.read_csv(SEL / "resume_par_groupe.csv")
base = pd.read_parquet(SEL / "base_selection.parquet")
display(resume_eco["statut"].value_counts().to_frame("codes ECO"))
print(f"Parties retenues (base unique) : {len(base):,}")
display(resume_eco[resume_eco.statut == "ok"].sort_values("n_retenues", ascending=False).head(20).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
base["ECO"].value_counts().head(30).plot.bar(ax=axes[0], title="Parties retenues par code ECO (30 premiers)")
base["rarete"].plot.hist(bins=40, ax=axes[1], title="Rareté des parties retenues (-log10 p)")
plt.tight_layout(); plt.show()
liens(base.sort_values("rarete", ascending=False).head(15)[
    ["Site", "ECO", "Result", "rarete", "rang_groupe", "features_extremes"]].round(2))

**Commentaires :**

*(vos observations ici)*

## Étape 3 : PU learning, des quelques milliers de candidates à 20 parties

* **Positifs (P)** : parties du mois Lichess (hors janvier et février 2026) ;
* **Non étiquetés (U)** : parties témoin tirées au hasard dans la même population que les candidates ;
* **À étiqueter** : les candidates de l'étape 2.

Les trois groupes sont réévalués par le même Stockfish, à la même profondeur, avant le calcul des features.

### 3a. Parties du mois (scraping du blog Lichess)

In [ ]:
PU.mkdir(parents=True, exist_ok=True)
if LANCER["scraping"]:
    if MODE == "exemple":                          # pas de réseau en mode exemple : parties du mois synthétiques
        shutil.copy(RACINE / "exemples" / "gotm_exemple.pgn", PU / "gotm.pgn")
        print("Mode exemple : exemples/gotm_exemple.pgn copié dans", PU / "gotm.pgn")
    else:
        lancer("scraping.py", "--dossier", DONNEES)

if (PU / "gotm_index.csv").exists() and MODE != "exemple":
    index = pd.read_csv(PU / "gotm_index.csv")
    print(f"Parties du mois récupérées : {len(index)} ({index['mois'].nunique()} mois)")
    liens(index[["mois", "Site", "White", "Black", "Result", "n_plies"]])
else:
    print("Parties du mois :", open(PU / "gotm.pgn", encoding="utf-8").read().count("[Event "))

### 3b. Groupe témoin

In [ ]:
if LANCER["temoin"]:
    lancer("temoin.py", "--dossier", DONNEES, "--n", P["n_temoin"], "--graine", P["graine"])
temoin = pd.read_parquet(PU / "temoin.parquet")
print(f"Parties témoin : {len(temoin):,}")
display(temoin.drop(columns=["Moves_clean"], errors="ignore").head())

### 3c. Augmentation : évaluation Stockfish et recalcul des features

In [ ]:
if LANCER["augmentation"]:
    args = ["--dossier", DONNEES, "--profondeur", P["profondeur"]]
    lancer("augmentation.py", *args, *(["--stockfish", STOCKFISH] if STOCKFISH else []))
aug = pd.read_parquet(PU / "features_augmentees.parquet")
display(aug.groupby("source").agg(parties=("cle", "size"), evaluees=("acpl_gagnant", "count")))
comparer = ["acpl_gagnant", "accuracy_gagnant", "n_gaffes_perdant", "win_min_gagnant", "n_sacrifices_pieces",
            "n_plies_sacrifice_valide", "n_brillants_gagnant", "swing_max", "n_echecs", "n_plies"]
display(aug.groupby("source")[comparer].median().T.round(2).rename_axis("médiane par groupe"))

**Commentaires :**

*(vos observations ici)*

### 3d. PU learning

In [ ]:
if LANCER["pu"]:
    lancer("pu_learning.py", "--dossier", DONNEES, "--seuil", P["seuil_pu"], "--graine", P["graine"])
r = json.load(open(PU / "pu_resume.json", encoding="utf-8"))
display(Markdown(f"### Nombre de candidates étiquetées positives : **{r['n_candidates_positives']}** "
                 f"sur {r['n_candidates']:,}"))
display(pd.Series(r).to_frame("valeur"))
display(Image(filename=str(PU / "pu_scores.png")))
if (PU / "pu_coefficients.csv").exists():
    display(pd.read_csv(PU / "pu_coefficients.csv").head(15).round(3))

**Commentaires :**

*(vos observations ici)*

### 3e. Échantillon final (20 parties)

In [ ]:
if LANCER["echantillon"]:
    lancer("echantillon_final.py", "--dossier", DONNEES, "--n", P["n_final"])
final = pd.read_csv(PU / "echantillon_final.csv")
liens(final[[c for c in ["rang", "profil", "Site", "ECO", "Result", "rarete", "p_pu", "features_extremes"]
             if c in final]].round(3))

**Avis sur les 20 parties :**

*(vos observations ici)*

## Comparaison : 200 parties quelconques (10 blocs de 20)

Tirage uniforme parmi **toutes** les parties du dump, sans filtre : une référence pour juger l'échantillon final.

In [ ]:
if LANCER["comparaison"]:
    lancer("comparaison.py", "--dossier", DONNEES, "--dump", DUMP, "--graine", P["graine"])
comp = pd.read_csv(DONNEES / "comparaison" / "comparaison_200.csv")
tableau = comp.pivot(index="position", columns="bloc", values="Site")
tableau.columns = [f"bloc {b}" for b in tableau.columns]
display(HTML(tableau.to_html(render_links=True)))

**Comparaison avec l'échantillon final :**

*(vos observations ici)*

## Conclusion

*(vos conclusions ici)*